# WEL01 · M10 통합 실행 결과와 최종 보완 사항 확인 (학생 실습용)

- 본 노트북은 **미션10(M10) 전용 학습자 실습용** 노트북입니다.
- 원자료 생성·참고 구현·상황판 소스가 노트북에 내장되어 있어 별도의 `.md` 파일을 업로드할 필요가 없습니다.
- 위에서부터 첫 실행 셀, 준비 셀, `STEP 01`~`05`의 `TODO`를 순서대로 완성한 뒤 마무리 셀을 실행하면 `WEL01/outputs/M10/WEL01_M10_result.html`이 생성됩니다.

In [ ]:
from pathlib import Path
import base64,zlib,json,re,html,hashlib,io,time,ast,copy
import numpy as np,pandas as pd
from IPython.display import display,HTML

PROJECT_ID='WEL01'
ROLE='student'
WORKDIR=Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT_HOME=WORKDIR/PROJECT_ID
OUTDIR=PROJECT_HOME/'outputs'
SOURCE_BUNDLE=None
DASHBOARD_ASSETS=None
REFERENCE_CACHE={}
REFERENCE_EXECUTION_SERIAL=0

def _sha(value):
    return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()

EMBEDDED_BUNDLE_B64 = '...'
EMBEDDED_ASSETS_B64 = 'eNqNOl1z20aSfwVxNgERgwgpiaIEHuLzeu0938ZOKvLePsgqGQSGJCIQ4GJASjqGW/Za2XIib61u177IieSS7+zE3kvqFH8qVd6X/BwC+g/XPTMAAYrM5cEiMNPT3dPf3XD/lEXpKf3UO/26v1Gkzr87XlOv+4FNgiKsDOq+vdlvm0HT8fRSrW5aa83A73q2/iZpNGYb8zXLd/1Af7NcnanPWrWG74V6udLZeLesVSrS2cAxXVW+7Ie+tGR6VPrNR7JK4aFISeA0BvVuGPqe6nidbqiGZCM0A2L2GRLHawFIKED6VjegQKfjO15IgoHmmpt+N+zbDu3Ao94MHLuGf4ohacNKSIrAWLftUb08O9PZkNqO1zY3CiW13AiUGrwVW8RptoDZUqnXGmg0JJ2iZ/b62SuWq7P1uUVxxfWWE5Jax7RtFFF5AZAudjZqHZ86oeN7Og0da22zFvodEJTpOk0PLuk2YN0MwtpkahINA99rpteou761xoSIqiCMSkqyJCHNmVJnI4NAiCePYN2xwxbSequGQi0ybnSXNMJaokqpJJXLGeRlFFIZcNe48nXYlKjvOrb05lx93qxWxEYxMG2nS/Uqgo5kFQag1Q6ozwuz8jrB6bIJJlEEbSKkgZsrOZGTcn1h3kytqjRjzy4mlMUiB8kKsW267nQZ4jVdxyOpxrU5IYYiKquC8rQAHBjqo2Fw6ZVS0cwAhDSLogELErvlSonpgWySeuCv9wVrc4vzi9WFLPEZJE5CsNkiiMdCfFppgbRTklKr3B/BI60xZmcridJA5Ki2UoZhqTWTPc2ZTDW8kIOczUCW5ydAUsvvkOQq89U5u7yY8/hGo1Gx6yOTKaHJzKUmk1hGBjVGAqk00NAzf46vjrxUq4KfJl47Mw+kuOc2zQ73iowGuUt0TI+4OQVmeOfOe9K07SopkXHTXsz6xeKIVt0HC24z8gMtNOsuKVIr8MH2/B4JGq6/rpvd0GdmItQ3V0HeGGx/ZMWu2aFETx6y7pq3nEHYUkO7P4Fti8yRhRPOnTBdFbGJ3ZrZHdE9fz0wO4Ax724LjVJjdnIQG4QY/KUw0L2wVbRajmsXSI94Sg5Do9owG3WQh++7dTNItdxwyQbTFiqLh0Ngpk11i2AEr+F+EVnS8U9qMDPcYAS2JLyNWeEERdbnrarVGFNkJaPICpfJCDVLOv3s9sLE6CcwCyXNsXDRJmHgWD8r+zQCiRlFIoo0t3JbGtEhM2SRWDkdlrL2MDsiLNVTi+MiNL3NdUiXJGsRARognOiGxO6PRSTh45V6tVy1wHVcP+xnrFAYL+M7H1YzXldmknD9Zj9rZp2AMJZqUxgci8sZV5lFV6nlPGmgeX5IspxlEzfLVgFhyOAUWK7p/lzLmMtYBo99oRl2aRL8ZiqV6lyVM7vOydV91x5oDR8kNXaHJGAuVK2F6kkJUbNH/r+qQnBd+gn7xdzPYsI/t4ntmIVRLlpAwSn9NIdlSxQReqfaZlpITYFgMT5XO2VKpZQQOs/8pLIklyqquWyUy3ogxsHgqnfV0xwPSqKuhcGIZgV9Mi8uVKbY2CCHRGrNJfXrzBxLoEwsORAen8d9LQdi+Tbp5+NpAGXHiSpkrNCYzmMqxRRgU9j8BNZ4suGq0hvOBrHHwVrq2II9gdtpzFz1TqmnPsY2oFBQjPf6cpcSLE0dK5TBTgGvZBnrjmf769rvzr9fKq+e++DyhYu/xspGMoFijxgltWM28afhuBDfDTk5iS5MjbNBYG4WKooG225BlhWxGzpgXU1qLK+IBeIahdBsqpZLWUeA/PCdDcP2rW4brEezoE0IyXmX4BuCKzWnUYAjyoZmuSall802MeAdlxHLG4YBvkcaoB0bYHDpHLdEYwnu6TUZlALhJOwGnrQxENyYnc6IapOEguQvNy/aBRk203tw3RgE7mY7PVmV+UK6D5pmm/ALm4nyYRv+apSEZ0NgA7yGAFqskF2zTlyAjP73KP50V4q2Hw+fbQlwoEs8yMWAjvcPsup1XVe1tLZDKahfSai2TcdjZPEBsAnvAzycvQQVYFURRKnBQrLIQRJU3KOZoq56ja7H7EwCi/Bc30QMbaL2TLdLFKEuc5q6ZBPom1orIA3jtx+9L3Y/qH9MrBDeCx5Zl37p+vXC8r8ufXBZo0w/TmOzwNDzq84oK2o/3OwQHbXgQuhHdt79mPqePFAQfcKZgZzBuwVAawWlBrK+4rQJ3IzZOjIQkJ6/lmGA86aokHFKyiBzW9ry16+gKxZ4p6OKOKlCC0CTe6NrZewgWyeOrB4XGRB7StdbBBhmy/gkK2oY8NeAgTBikIOC86bVKqwZ74VB1hjCljCENQVEgBiS3RAqZ0YpWcDNVLNQ5XEq+ASEgECBbwWS35Cyl4PnEUMjuDWEE/wloD0jWF5bqcGJHI+24LFnGAY+fPIJPqXOeQYt/vje3Xj/SNZRwYAYAWSfqUc+M24Tit5TFDamSMgARWWQuy3uKjVUTCoQ3FZqXI3JIgLk9M2rrUTZa2pvXAxcxRwMBDJ2Vwi9XqoRFVfqyeWVcdqM6Qxp2INCoJAQpIaFJUqT0GUeb1dqwiFTcyjUVQeDJUQ8BwTGwZT6pOAi2m8RiIBx4kLAr4MjtCFDTINWBkoNgwSAQZVgkXPYEcBOQSxnbs7lIppjWb32i76ldQIfNXjRHkg/vpJwRUSr9L3ldwM6iLf3hodbuBZvP4wP9qQPN8MWSATejv/zM2n49HD47PU1Ls9WOQ19oROCRtlqJ12k0GzAHXxlGous5YXf4YudeO9B/OUT6fjuk3jreym+vxP9923GxTdbEH2l+C787LKFw6PhswMp+vvj6M9Hgi1gCU7eiz9/AQfj/V0JVG+bVHAbPXotDV98G73cAuzR9q1o+6GGC/GDQ4C4Hn+9H+9fl+IXe/F/fSp9eHZpSTq+vRMfgBM853TviUNyGtmxXsuYIL5CuMAuMF1VVNYCjN7FUdYos1VKmLHBebYmo1HCr2aboQlmo3EbWgIbSbIkXzldTiAzAm3NCLFfW7py/kNQ6PgR6NDtJRyGFWZUuQTmJAEQ5YobCIUKh8KehZkNBSPx4I0OQHhSdHgIUmE6uLnPtPTFc4YDojk0c+dYAGLmFH+1g8aCe2CB56DwDgfDwz0pevY8fvD8msLqBaq5xAw8EvxL2HYTT8tUUWMyytZXIKrMK5ReiOXKpfeNHM68kDIHlAEnxmslYhuMTarx90JgvPcGf/zkEyHFYFnmd7r4K3lFAYKW27UJLXCw1CywuaXZ7MObXbAFaHfZOmt7GfsdjblxCxobrNcS9Mwib34mM4ifqEzG4TlKlqMNzhVb8D2rZXpQGbLKUhSHKWSNF421JNwldRe0kbxe4lFOWJYMDhEfbElAF27kQcU2AQb8JN6/zWFqiEeDFhbDvW0wYgaQw6Oj5QKug4G+M1t6z0hUAnr0mmGLY4BLYPnA74DQxWKGZYbtBMjp0xkQppeRIXRURMtuoI4yRcbuEQGwc7o8+MMv+pfMsIV+UMgwqo7xCd70Lpj72OoApHBNGXNWxgsUQpliBnZVqiXVTIqEwo1IQfCiZqgrE/y/w9IJdtGpcfz4Cn6jZ0fDw7/++Cq6tRUf3BARkXluLgJK8b0tiLgY4TCKJZgZmcS6KSQIi0wJXXwzF5FmU7v5akcE9Oj2daQpxU//R5pnNpKVBMeBiYM9nBMVF5dDSV1U0q2P0F2T9fkxpjmIkmmApvCMW5M5Hj6/Hh9ySe1DutmShkeH0YP9UQJIvpbwcky8AGpECZiTlTEf3zo4vrl/fHdPGr56Pfz+UGXp6/Zj0E5SdUnR0zvx3pYqJX7EVpGRaPt5tP1DvJ/wcnx3N946ir+8o8kjctz/Wc+XVirppu+x6MN8JAdi5M/n5JJspUZg9kjO7WVcwRzOuJKwQJTig+vx/UdoFbCX8820Y7nGO9lsIbIaEMsPbA3biGtqP61Y9Ez1ogpgPT2mcnVfgDBNgg5E6xD2GqM3lV1V7TqXiEm7AWuCqC4aX7VNTA9HJHJaPkhc90LKEPKiJ4dgBbtovuA9kAF/fAX1CkCJMkHUJtD45ASHV8/bJe6mYuy222awOc2Z+O4U20yLpfjmjeN7T4ALtEtRLYujqny8uxXf/zZpXTGqJaUAxqrKtZMnRm6Knpn3NRHmTpwZxRNZzRcCE/Bz5Lx4kFk6iZ7eAhPfHT79R/zFTry1l1R38T/uRH/bE9WdPAGVAONFafz8lqxe7rbrEO0tHNKaUCQ3IIZ6FlkF6fqeTRUt9C/g6KYwq4ymBpBLqdE3Pb9tupurFhYqOrjpa8YKZLubN4DG3vEXu1J8axdEAg08dJIpINSIACGCWhawGZg2WUUqDtT4nrWpy9Hfvh8eAa/7r9HAUAABcU3UyGrDDMD2bV6c6PKFsx9Jw8Od+OXd6NEBgDl0DV3DdF0gubfFiNz/FvW791hW2Twq3QerjR9CMXsCrNuBspKsor2vtqkui7J+ePh9vH0gHf/ls+GrA6lNR4DmxgQ4KJLhtgyOZ1cGI7QxfHUUfQ2B6uB29Oc70Tc3GBgbXAODicgOH4NFRjsg3i/vxE+/4+Iyu7YTrkLwcezVOoGWioC8nnGpQsERPdgD3CJTyWpotjtA2SYhOA6xU1Dcf7mX7qdfVVJUe1vxD0CtUV5tQjuy2umArMtwtT0IzNJbkEEtaBscdn3B7vHNHSiupOOvtqB0TWygRwJIxXCXb66j+x/ffQwbiS1GP3B5Wy1iraW3hq7kKZw/2AEe4cB30d+fcGwdk1K8whjEn14wr0IIMI7U0iBlY3MGBiKnpRoJ8JOOgWObJRIWluUTtgeuNtnUcGNkXPCWtSU2r+JXxR2oYleyM4flNZW5zwrOHvjwRgM+AgeqYu5XymgUwZ0S5xFQ9+cGDnz+gIFNQeeCboMkERLKAs8Q19NaJi2sKWcKwst7yjvlUmnk02XltPxWfmLhMUj57bff4Gc0h170QtJkp8/00qMzSu5YHcoz8BH5TKF3Ro53b8kgfIzvt+Ldh7Ki98ZDEburyphVBhPidmdUWIrQhO3dKut8IRCxHjFNGfx4miMQZlqGwL0p+YEXdLyQiO9dl6I/PsYGV4x10iZ9jK22bxN31eyCuwe05XSU3ISJbKKel2X8/x9gYMjNqgs5FKcYzG/5UtLQi7SKLRBLtKtdj4EmFpUCcs9nVdOKgkNkYSlkc0U5cccTTHO4CdLIQMq/vSiSIMYSSDYSH3Gwnp+POI5vPI++/g69XIqPHoCvQW2FWT9T0kB6ElVNZoaQS1Y4dohf3okeHEB9IKIlI8SLt4fRI4hgh3+N73+amSbk1c9sopZKQWMxhL79du41ScX9pBClIZ1iJGxvSkmeJFCesyDzb++dKHKhS2KY4RdH6j7oc2z+l+MsmQ2PEYbTYxdl28oApybptBvqJD4wyc+KEEaIBD+Tjt+erU0bpjD4n749huyk8mBlCS/OMY/JOQeggcWGrJyLdFrRbvKuvo2mjlvYh7ebGoAb8I89m25oJNVYnjBYTDLv2j443oXif3cn+vyOBCvR548SPvIXAYzKICshzpCYaICOpkgDdibLIs8ZZBhYSJMZS7xJyBhZgVA7DW38RJHXGACkEQa/10LpeelXEuTK6NsjNmfM1OaDzJwRXCt69h/QCr28jiUGeCFLuV/8CdogbKJzH8D+7eL5360uXTl75bzRP9kRjAZnelLzqskcSh+NpNRsuarnq1coqwIaYumri+FQ0m3OKlN6iUENJ8hgMfhZzjFKNeefKjXn9OnEWOrZtkmp1dM5H81M+Bwc7tWnjLHHRnvOpKmemOSxz6GjQSyfXTsrYk6LJHITExFLMN028FslFM6a568XlJr4vugko6LR98VkvAIPv++CR5/1nDbLAhcCs03Y553pO30hNK3Tpa1CH0Wgw32Ad/w/OfYlqp9gpRiqLELqMmO8GPrFcN0vmgnyYgOxU+zDptiKlleckShOUQbpPJ/xU8996asrg/SuAwX+XPVODf4PT6yS7w=='

def _decode_b64(payload):
    return json.loads(zlib.decompress(base64.b64decode(payload, validate=True)))

def load_embedded_bundle():
    global SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE, REFERENCE_EXECUTION_SERIAL
    bundle = _decode_b64(EMBEDDED_BUNDLE_B64)
    loaded_assets = _decode_b64(EMBEDDED_ASSETS_B64)
    if bundle.get('projectId') != PROJECT_ID or bundle.get('version') != 'wel01-integrated-2':
        raise ValueError('프로젝트·버전 불일치')
    if bundle.get('fingerprint') != _sha({k: v for k, v in bundle.items() if k != 'fingerprint'}):
        raise ValueError('원자료 지문 불일치')
    if bundle.get('assetFingerprint') != _sha(loaded_assets):
        raise ValueError('화면 소스 지문 불일치')
    for m_id, item in bundle['missions'].items():
        if len(item['stages']) != 5:
            raise ValueError(m_id + ' 5단계 누락')
        if sum(s['minutes'] for s in item['stages']) != item['minutes']:
            raise ValueError(m_id + ' 시간 불일치')
        if hashlib.sha256(item['referenceCode'].encode()).hexdigest() != item['referenceCodeSha256']:
            raise ValueError(m_id + ' 참고 코드 지문 불일치')
    namespace = {}
    exec(compile(bundle['runtimeSource'], 'WEL01_reference_runtime.py', 'exec'), namespace)
    bundle['_reference_runner'] = namespace['_reference_table']
    SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE = bundle, loaded_assets, {}
    REFERENCE_EXECUTION_SERIAL = 0
    print(f'WEL01 M10 내장 교육 자료 연결 완료 (5단계 · {bundle["missions"]["M10"]["minutes"]}분)')
    return bundle

def _connected():
    if SOURCE_BUNDLE is None:raise RuntimeError('먼저 첫 실행 셀을 실행해 내장 교육 자료를 로드하세요.')

def _parameter_code(mid,code,parameters):
    allowed=SOURCE_BUNDLE['missions'][mid]['parameters']
    if set(parameters)-set(allowed):raise ValueError('정의되지 않은 변경 파라미터')
    if mid=='M02':
        value=parameters['FLATLINE_RUN']
        if not isinstance(value,int) or value<1:raise ValueError('고착 분은 양의 정수')
        code=code.replace('def flatline(series, run=8):',f'def flatline(series, run={value}):')
    elif mid=='M05':
        value=float(parameters['SCORE_THRESHOLD'])
        if not 0<=value<=1:raise ValueError('문턱은 0~1')
        code=code.replace('eng = (test["score"] >= 0.5).values',f'eng = (test["score"] >= {value!r}).values')
    elif mid=='M09':
        value=float(parameters['MIN_RECALL'])
        if not 0<=value<=1:raise ValueError('재현율 하한은 0~1')
        code=code.replace('(g["재현율"] >= 0.90)',f'(g["재현율"] >= {value!r})')
    inserts={}
    for node in ast.parse(code).body:
        if isinstance(node,ast.Assign):
            names=[]
            for target in node.targets:
                names.extend(x.id for x in ast.walk(target) if isinstance(x,ast.Name))
            updates=[f'{key} = {parameters[key]!r}' for key in names if key in parameters and key not in ['MIN_RECALL','SCORE_THRESHOLD','FLATLINE_RUN']]
            if updates:inserts[node.end_lineno]='\n'.join(updates)
    lines=code.splitlines()
    for pos,value in sorted(inserts.items(),reverse=True):lines.insert(pos,value)
    return '\n'.join(lines)+'\n'

def source_table(mid,parameters=None,refresh=False):
    global REFERENCE_EXECUTION_SERIAL
    _connected();item=SOURCE_BUNDLE['missions'][mid]
    p=dict(item['parameters']);p.update(parameters or {})
    key=(mid,_sha(p))
    if refresh or key not in REFERENCE_CACHE:
        tuned=None
        if mid=='M10':
            # 현재 M9 실행 결과를 우선 사용  아직 실행하지 않았으면 기본 M9를 실행한다.
            matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            if not matches:
                source_table('M09');matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            m9=max(matches,key=lambda x:x[1][1]['executionOrder'])[1][1]['selected']
            tuned={'heat_temp':float(m9['th_heat']),'cold_temp':float(m9['th_cold']),
                'min_persist':int(m9['persist']),'wind_corr':float(m9['corr'])}
        code=_parameter_code(mid,item['referenceCode'],p)
        raw,meta=SOURCE_BUNDLE['_reference_runner'](mid,code,tuned)
        REFERENCE_EXECUTION_SERIAL+=1
        meta['executionOrder']=REFERENCE_EXECUTION_SERIAL
        meta.update(parameters=p,md_fingerprint=SOURCE_BUNDLE['fingerprint'],reference_name=mid+' 교육용 참고 구현')
        REFERENCE_CACHE[key]=(raw,meta)
        if mid=='M09':
            for stale in [k for k in REFERENCE_CACHE if k[0]=='M10']:del REFERENCE_CACHE[stale]
    raw,meta=REFERENCE_CACHE[key]
    out=raw.copy(deep=True);out.attrs['reference_meta']=copy.deepcopy(meta)
    return out

def TODO(mission,step,instruction):
    raise NotImplementedError(f'{mission} STEP {step}: TODO를 실제 pandas 코드로 완성하세요. {instruction}')

def _rows(df):
    return json.loads(df.to_json(orient='records',force_ascii=False,date_format='iso'))

def finish_mission(mid,raw,stages,stage_specs):
    _connected();canonical=SOURCE_BUNDLE['missions'][mid]['stages']
    if stage_specs!=canonical:raise ValueError(mid+' 내장 기준과 노트북 STEP 정의 불일치')
    if len(stages)!=5 or not isinstance(raw,pd.DataFrame) or raw.empty:raise ValueError('5단계와 원자료 필요')
    expected=list(raw.columns);original_ids=raw['원행ID'].tolist();report=[];cumulative=[]
    for k,(df,spec) in enumerate(zip(stages,stage_specs),1):
        if not isinstance(df,pd.DataFrame):raise TypeError(f'{mid} STEP{k}: DataFrame 필요')
        col=spec['outputColumn'];cumulative.append(col)
        if df['원행ID'].tolist()!=original_ids or not df.index.equals(raw.index):
            raise ValueError(f'{mid} STEP{k}: 행 누락·재정렬·중복이 발생했습니다.')
        if any(x not in df.columns for x in expected+cumulative):raise ValueError(f'{mid} STEP{k}: 필수 열 누락')
        if not df[expected].equals(raw[expected]):raise ValueError(f'{mid} STEP{k}: 원자료를 변경했습니다.')
        if df.columns.duplicated().any():raise ValueError('열 이름 중복')
        preview=df[['원행ID']+cumulative]
        report.append(dict(step=k,title=spec['title'],goal=spec['goal'],outputColumn=col,
            learnerHtml=spec.get('learnerHtml',''),rowCount=len(df),columns=list(preview.columns),rows=_rows(preview),minutes=spec['minutes']))
    location=OUTDIR/mid;location.mkdir(parents=True,exist_ok=True);stem=f'WEL01_{mid}'
    stages[-1].to_csv(location/(stem+'_analysis.csv'),index=False,encoding='utf-8-sig')
    meta=copy.deepcopy(raw.attrs['reference_meta']);plots=meta.pop('plots',[])
    details=dict(projectId='WEL01',mission=mid,originalRows=len(raw),codeSteps=report,
        referenceMeta=meta,mdFingerprint=SOURCE_BUNDLE['fingerprint'],
        structuralValidation='원행과 원본값·누적 열 검증 완료  코드의 의미·교관 PASS 판정과 별개')
    (location/(stem+'_code_report.json')).write_text(json.dumps(details,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    config=dict(projectId='WEL01',mission=mid,title=SOURCE_BUNDLE['missions'][mid]['title'],
        scenario=SOURCE_BUNDLE['missions'][mid]['scenario'],hours=SOURCE_BUNDLE['missions'][mid]['minutes']/60,
        stages=report,sourceColumns=list(raw.columns),sourceRows=_rows(raw),meta=meta,plots=plots,
        fingerprint=SOURCE_BUNDLE['fingerprint'],role=ROLE)
    safe=json.dumps(config,ensure_ascii=False,default=str).replace('<','\\u003c')
    document='<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>'+stem+'</title><style>'+DASHBOARD_ASSETS['css']+'</style></head><body><div id="app"></div><script>window.WEL01_CONFIG='+safe+';</script><script>'+DASHBOARD_ASSETS['js']+'</script></body></html>'
    path=location/(stem+'_result.html');path.write_text(document,encoding='utf-8')
    if not globals().get('HEADLESS_VALIDATION',False):
        display(HTML('<iframe title="'+stem+'" sandbox="allow-scripts allow-downloads" style="width:100%;height:1150px;border:0" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
    print('HTML:',path);print('행 보존·누적 열 검사 완료  최종 PASS는 교관 판정')
    return path

load_embedded_bundle()

## M10 통합 실행 결과와 최종 보완 사항 확인

현재 M09에서 고른 설정으로 통합 예시 프로그램을 실행하면 10개 자체 점검이 나옵니다. 결과와 근거, 실제 전달된 설정값을 확인하고 아직 남은 보완을 정리하세요. 자체 점검 통과를 교관 평가나 실제 운용 승인으로 대신하지 않습니다.

실습 시간 210분. 준비 셀 → STEP별 설명과 코드 → 마무리 셀 순서로 실행합니다.


In [ ]:
MID = 'M10'
PARAMETERS = {}
raw = source_table(MID, parameters=PARAMETERS, refresh=True)
print("입력 표의 행·열 수:", raw.shape)
display(raw.head(6))


### M10 STEP 01 — 통합 실행의 자체 점검 결과 읽기

<h4>이 단계에서 해결할 일</h4><p>통합 예시 프로그램은 10개 자체 점검 결과를 만듭니다. 먼저 실제 실행의 결과를 읽되 프로그램의 True가 교관의 최종 합격 판정과 같다고 받아들이지 않습니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m10_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>raw</code>, 완성한 결과 표는 <code>step1</code>입니다. 새로 만들 열은 <code>자체점검_통과</code>이며 배정 시간은 42분입니다. 노트북의 <code>M10 STEP 01</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>실행결과</code> — 현재 통합 실행이 각 점검에 대해 남긴 불리언 결과입니다. 문자열이 아닌 True·False여야 합니다.</li></ul><h4>수행 순서</h4><ol><li>raw의 점검 이름과 실행결과를 확인합니다.</li><li>실행결과가 실제 True이면 자체점검_통과=True, False이면 False로 표시합니다.</li><li>새 열을 추가한 표를 step1에 저장하고 전체 10건과 통과 수를 기록합니다.</li><li>실패 점검은 이름과 현재 결과를 적습니다. 결과를 바꾸기 전에 실행 근거를 다음 단계에서 확인합니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 해당 코드의 자체 점검을 통과했다는 뜻입니다. 교육 평가 합격이나 실제 운용 허가가 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"실행결과": true}</code></td><td><code>True</code></td><td>실제 불리언 True입니다.</td></tr><tr><td><code>{"실행결과": false}</code></td><td><code>False</code></td><td>실패 결과를 유지합니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>문자열 "True"나 누락값은 올바른 불리언 입력이 아닙니다. 입력을 바로잡은 뒤 다시 실행하며 참으로 임의 변환하지 않습니다.</p>

In [ ]:
# M10 STEP 01 — 통합 실행의 자체 점검 결과 읽기
step1 = TODO(MID, 1, '통합 실행의 자체 점검 결과 읽기')
display(step1[["원행ID", '자체점검_통과']].head(10))


### M10 STEP 02 — 각 점검에 실행 근거가 남아 있는지 확인하기

<h4>이 단계에서 해결할 일</h4><p>결과가 True여도 어떤 값을 확인했는지 기록이 없으면 검토할 수 없습니다. 점검별 JSON을 읽어 근거 항목이 존재하는지 먼저 확인하고 내용의 타당성은 따로 확인합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m10_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step1</code>, 완성한 결과 표는 <code>step2</code>입니다. 새로 만들 열은 <code>검증_근거존재</code>이며 배정 시간은 42분입니다. 노트북의 <code>M10 STEP 02</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>실행근거</code> — 측정값·비교값 등을 JSON 객체 문자열로 저장한 열입니다. {}는 근거 항목이 없는 상태입니다.</li></ul><h4>수행 순서</h4><ol><li>step1의 실행근거를 json.loads 등으로 읽습니다.</li><li>객체가 비어 있지 않으면 검증_근거존재=True, 빈 객체이면 False로 표시합니다.</li><li>새 열을 넣은 표를 step2에 저장합니다. JSON을 읽지 못하면 멈춘 원행과 원문을 기록합니다.</li><li>True인 건도 항목 이름·값·단위·비교 기준을 직접 확인합니다. 값이 0 또는 False라는 이유로 근거가 없다고 처리하지 않습니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 근거 항목이 있다는 뜻입니다. 그 근거가 올바른 계산이거나 목표를 충족했다는 뜻은 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"실행근거": "{}"}</code></td><td><code>False</code></td><td>근거 항목이 없습니다.</td></tr><tr><td><code>{"실행근거": "{\"FAR\":0}"}</code></td><td><code>True</code></td><td>0이라는 측정값도 근거입니다.</td></tr><tr><td><code>{"실행근거": "{\"passed\":false}"}</code></td><td><code>True</code></td><td>실패를 나타내는 근거도 존재합니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>오류가 난 JSON을 빈 객체로 바꿔 계속하지 않습니다. 지정 입력은 객체이며 목록 등 다른 자료형이면 입력을 확인받습니다.</p>

In [ ]:
# M10 STEP 02 — 각 점검에 실행 근거가 남아 있는지 확인하기
step2 = TODO(MID, 2, '각 점검에 실행 근거가 남아 있는지 확인하기')
display(step2[["원행ID", '검증_근거존재']].head(10))


### M10 STEP 03 — 실패했거나 근거가 없는 점검 표시하기

<h4>이 단계에서 해결할 일</h4><p>점검이 실패한 경우와 결과만 있고 근거가 없는 경우는 모두 후속 확인이 필요합니다. 두 사유를 묶되 왜 미완료로 분류했는지 따로 남깁니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m10_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step2</code>, 완성한 결과 표는 <code>step3</code>입니다. 새로 만들 열은 <code>미완료_존재</code>이며 배정 시간은 42분입니다. 노트북의 <code>M10 STEP 03</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>자체점검_통과</code> — STEP 1의 실제 통과 여부입니다.</li><li><code>검증_근거존재</code> — STEP 2의 근거 항목 존재 여부입니다.</li></ul><h4>수행 순서</h4><ol><li>step2에서 두 표시를 같은 행에서 읽습니다.</li><li>통과가 False이거나 근거 존재가 False이면 미완료_존재=True로 표시합니다.</li><li>둘 다 True일 때만 False로 하여 step3에 저장합니다.</li><li>미완료 건을 실행 실패·근거 누락·둘 다로 나눠 기록하고 재실행 또는 추가 확인할 항목을 적습니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 후속 확인 대상이라는 뜻입니다. 미완료를 정확하게 표시한 학습자의 계산은 정답일 수 있습니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"자체점검_통과": false, "검증_근거존재": true}</code></td><td><code>True</code></td><td>실패를 뒷받침하는 근거가 있습니다.</td></tr><tr><td><code>{"자체점검_통과": true, "검증_근거존재": false}</code></td><td><code>True</code></td><td>성공 주장에 근거가 없습니다.</td></tr><tr><td><code>{"자체점검_통과": true, "검증_근거존재": true}</code></td><td><code>False</code></td><td>두 확인 항목은 충족했습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>근거가 있는 실패를 완료로 처리하지 않습니다. 표시가 False인 건도 근거 내용의 수치 검토가 남을 수 있습니다.</p>

In [ ]:
# M10 STEP 03 — 실패했거나 근거가 없는 점검 표시하기
step3 = TODO(MID, 3, '실패했거나 근거가 없는 점검 표시하기')
display(step3[["원행ID", '미완료_존재']].head(10))


### M10 STEP 04 — 통합 실행이 현재 M09 설정을 사용했는지 확인하기

<h4>이 단계에서 해결할 일</h4><p>보정한 설정이 통합 실행으로 넘어가지 않으면 같은 프로젝트라도 다른 조건의 결과를 보고하게 됩니다. 설정 출처 표시를 계산하고 실제 전달된 값도 별도로 확인합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m10_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step3</code>, 완성한 결과 표는 <code>step4</code>입니다. 새로 만들 열은 <code>설정_연계확인</code>이며 배정 시간은 42분입니다. 노트북의 <code>M10 STEP 04</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>설정출처</code> — 현재 실행이 사용한 설정의 출처입니다. 기대 문구는 M09 현재 실행 설정입니다.</li><li><code>reference_meta</code> — 노트북 raw.attrs에 있는 실제 실행 기록입니다. M09 선택 설정과 M10에 전달한 값을 확인할 때 사용합니다.</li></ul><h4>수행 순서</h4><ol><li>M09를 먼저 실행한 뒤 M10 준비 셀을 실행합니다. M09를 다시 실행했다면 M10도 처음부터 다시 실행합니다.</li><li>step3의 설정출처가 M09 현재 실행 설정과 정확히 같은지 비교합니다.</li><li>같으면 설정_연계확인=True로 만들어 step4에 저장합니다.</li><li>문구 확인 외에 M09가 선택한 열·한랭 기준, 지속 시간, 풍속 보정값을 M10 실행 기록과 비교해 일치 여부를 적습니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 출처 문구가 맞는다는 1차 확인입니다. 문구만으로 모든 구성 요소의 연결이 증명되는 것은 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"설정출처": "M09 현재 실행 설정"}</code></td><td><code>True</code></td><td>기대한 출처 문구입니다.</td></tr><tr><td><code>{"설정출처": "기본 설정"}</code></td><td><code>False</code></td><td>현재 보정 결과를 사용했다는 표시가 아닙니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>출처 문구를 직접 고쳐 True로 만들지 않습니다. 설정값이 다르면 현재 실행 순서와 캐시 갱신을 확인합니다.</p>

In [ ]:
# M10 STEP 04 — 통합 실행이 현재 M09 설정을 사용했는지 확인하기
step4 = TODO(MID, 4, '통합 실행이 현재 M09 설정을 사용했는지 확인하기')
display(step4[["원행ID", '설정_연계확인']].head(10))


### M10 STEP 05 — 최종 보완 대상과 남은 확인을 정리하기

<h4>이 단계에서 해결할 일</h4><p>전체 결과를 넘길 때에는 실패·근거 누락·설정 연결 미확인을 숨기지 않고 한 목록으로 정리해야 합니다. 모든 표시가 양호해도 이번 실습이 확인하지 않은 범위를 함께 남깁니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m10_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step4</code>, 완성한 결과 표는 <code>step5</code>입니다. 새로 만들 열은 <code>최종_검토필요</code>이며 배정 시간은 42분입니다. 노트북의 <code>M10 STEP 05</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>미완료_존재</code> — STEP 3의 실패 또는 근거 누락 표시입니다.</li><li><code>설정_연계확인</code> — STEP 4의 현재 M09 설정 출처 확인 표시입니다.</li></ul><h4>수행 순서</h4><ol><li>step4에서 미완료가 True인지 또는 설정 연결이 False인지 확인합니다.</li><li>둘 중 하나라도 해당하면 최종_검토필요=True로 만들고 step5에 저장합니다.</li><li>True인 건마다 원행ID·점검 이름·실패 값 또는 누락 근거·다음 담당 역할·재확인 시점을 적습니다.</li><li>모두 False여도 자체 점검 수, 지정 9개 함수의 커버리지 범위, 실제 위험 재현율·FAR, 아직 확인하지 않은 운용 범위를 적습니다. HTML 기록을 저장한 뒤 최종 제출물을 확인합니다.</li></ol><h4>결과를 읽는 방법</h4><p>False는 이 두 표시에서 추가 사유가 없다는 뜻입니다. 교관 판정, 실제 의료 판단, 실전 운용 승인을 대신하지 않습니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"미완료_존재": true, "설정_연계확인": true}</code></td><td><code>True</code></td><td>점검 미완료가 남습니다.</td></tr><tr><td><code>{"미완료_존재": false, "설정_연계확인": false}</code></td><td><code>True</code></td><td>현재 설정 연결이 확인되지 않았습니다.</td></tr><tr><td><code>{"미완료_존재": false, "설정_연계확인": true}</code></td><td><code>False</code></td><td>두 표시상 추가 사유가 없습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>커버리지를 프로젝트 전체 검증률로 쓰지 않습니다. 기존 화면의 PASS 문구를 그대로 교관 판정으로 옮기지 않습니다.</p>

In [ ]:
# M10 STEP 05 — 최종 보완 대상과 남은 확인을 정리하기
step5 = TODO(MID, 5, '최종 보완 대상과 남은 확인을 정리하기')
display(step5[["원행ID", '최종_검토필요']].head(10))


### 마무리: 결과 화면 생성과 기록 저장

아래 셀을 실행하면 다섯 단계의 실제 계산 결과가 HTML로 저장됩니다. 왼쪽 STEP 메뉴로 결과를 확인하고 관찰 기록을 입력하세요. 기록 JSON을 내려받아야 입력한 글을 별도로 보관할 수 있습니다. 구조 검사 완료는 교관의 최종 합격 판정이 아닙니다.


In [ ]:
STAGE_SPECS = SOURCE_BUNDLE["missions"][MID]["stages"]
result_html = finish_mission(MID, raw, [step1, step2, step3, step4, step5], STAGE_SPECS)
